# 1 · Data Engineering

**Multimodal Recommender System for E-Commerce Item Cold-Start**

How 43.4 million raw interactions became a validated experimental corpus of
9,503 products, and how the cold-start condition was constructed and verified.

This notebook covers the **Final Data** section of the report and stages one and
two of the process flowchart. Every figure is saved to `results/figures/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = Path.cwd().parent
REPORTS = PROJECT / "reports"
TABLES = PROJECT / "results" / "tables"
FIGURES = PROJECT / "results" / "figures"
PROCESSED = PROJECT / "data" / "processed"
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "legend.frameon": False,
})

BLUE, ORANGE, GREEN, RED = "#4c72b0", "#dd8452", "#55a868", "#c44e52"


def read(name, folder=REPORTS):
    path = folder / name
    return pd.read_csv(path) if path.exists() else None


print("project:", PROJECT.name)

## 1. The raw data

Amazon Reviews 2023, Electronics category. Two files were required: the
interactions and the item metadata.

Two decisions here shaped everything downstream.

**The ratings-only export was used rather than the full review file.** This
project derives its text from product metadata — titles, feature bullets,
descriptions — and never from review text. The full review export is tens of
gigabytes, almost entirely prose that would go unread. The ratings-only export
carries exactly the four fields needed.

**The 0-core release was used rather than the 5-core.** The 5-core release
removes every user and item with fewer than five interactions before the data is
seen. Any independent analysis of interaction thresholds would then be measuring
the publishers' filtering decisions rather than the underlying distribution.

In [ ]:
sources = pd.DataFrame([
    {"file": "Electronics ratings (0-core)", "size": "925 MB",
     "records": "43,365,426 interactions",
     "fields": "user_id, parent_asin, rating, timestamp"},
    {"file": "Electronics metadata", "size": "1.22 GB",
     "records": "1,610,012 items",
     "fields": "title, features, description, categories, images"},
])
sources

**Products are identified by `parent_asin`.** Amazon assigns each colour, size
and storage variant its own identifier; the parent identifier unifies them.
Joining on the variant identifier would fragment each product family across many
sparsely observed records, starving the collaborative signal while appearing to
enlarge the catalogue.

The choice is verified downstream: the converged corpus shows a median of 16
interactions per item. A variant-level join would collapse that toward two or
three.

## 2. Assigning products to domains

The corpus is restricted to five structurally adjacent electronics
sub-domains, so that cross-domain purchasing behaviour exists to learn from.

The `categories` field is a **hierarchical path** running from broad to
specific, not a flat set of tags. This property governs the entire assignment
design.

In [ ]:
vocabulary = read("category_vocabulary.csv")
if vocabulary is not None:
    print("distinct category tokens:", f"{len(vocabulary):,}")
    display(vocabulary.head(12))

### A failure worth recording

The first implementation excluded any item whose category path contained an
accessory marker. It rejected almost the entire catalogue: **fourteen laptops
survived out of thirty-seven thousand**, and the domain imbalance reached
2,536 : 1.

The cause is that `computers & accessories` is the common ancestor of both
laptops and laptop bags. Excluding on ancestors removes the devices along with
the accessories.

**The fix is that the final token of the path is what the item is.** A laptop's
path terminates in `traditional laptops`; a laptop bag's terminates in
`bags, cases & sleeves`. Same ancestor, opposite leaves.

The pipeline reported success throughout. It was caught by a domain-count check
and a manual audit of twenty-five sampled titles per domain — aggregate counts
detect that something is wrong, reading records tells you what.

### The three-stage filter

| stage | operates on | purpose |
|---|---|---|
| 1 · leaf exclusion | last path token | remove accessories, parts, consumables |
| 2 · title exclusion | product title | catch accessories filed under a device leaf |
| 3 · title affirmation | product title | require a domain-relevant term |

Stage three is the robust one. The source metadata contains outright
misclassification — a toothbrush filed under loudspeakers — which no category
rule can anticipate but which fails an affirmative title test immediately.

Over-rejection was treated as acceptable throughout: the surviving pool exceeds
the corpus requirement by more than an order of magnitude, so a false rejection
is nearly free while a false acceptance contaminates the corpus.

In [ ]:
counts = read("domain_counts.csv")
if counts is not None:
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    order = counts.sort_values("n_items")
    ax.barh(order["domain"], order["n_items"], color=BLUE, height=0.62)
    for y, value in enumerate(order["n_items"]):
        ax.text(value * 1.02, y, f"{value:,}", va="center", fontsize=8)
    ax.set_xlabel("Items assigned")
    ax.set_title("In-scope products by domain, after three-stage filtering")
    ax.set_xlim(0, order["n_items"].max() * 1.15)
    fig.savefig(FIGURES / "domain_assignment.png")
    plt.show()
    print(f"total in scope: {counts['n_items'].sum():,} of 1,610,012 items")

## 3. The feasibility census

The design depends on customers who purchase across several of these domains. If
too few existed, the project would have failed months into implementation.

Rather than assume, the population was counted before any threshold was
committed to.

In [ ]:
census = read("census_users_by_domain.csv")
if census is not None:
    census = census.rename(columns={census.columns[0]: "domains_touched"})
    census["share"] = census["n_users"] / census["n_users"].sum()

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.3))

    axes[0].bar(census["domains_touched"], census["n_users"], color=BLUE, width=0.6)
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Domains purchased across")
    axes[0].set_ylabel("Users (log scale)")
    axes[0].set_title("Cross-domain purchasing is rare")
    for x, y, s in zip(census["domains_touched"], census["n_users"], census["share"]):
        axes[0].text(x, y * 1.35, f"{s:.1%}", ha="center", fontsize=8)

    axes[1].plot(census["domains_touched"], census["median_interactions"],
                 marker="o", color=GREEN, linewidth=1.6, markersize=5)
    axes[1].set_xlabel("Domains purchased across")
    axes[1].set_ylabel("Median interactions per user")
    axes[1].set_title("Cross-domain users are far more active")
    for x, y in zip(census["domains_touched"], census["median_interactions"]):
        axes[1].text(x, y + 0.5, f"{y:.0f}", ha="center", fontsize=8)

    fig.tight_layout()
    fig.savefig(FIGURES / "feasibility_census.png")
    plt.show()
    display(census[["domains_touched", "n_users", "share",
                    "median_interactions"]].round(3))

**111,270 users purchase across three or more domains.** The constraint was
therefore retained at three, and the contingency of relaxing it to two was never
needed.

The right-hand panel is the scientific justification for the constraint. Median
interactions climb monotonically — 1, 2, 4, 7, 14 — as domains touched
increases. The filter selects for interaction density rather than merely
reducing the size of the dataset.

## 4. Iterative filtering to convergence

Two constraints interact. Removing casual purchasers starves items of support;
removing those items disqualifies further purchasers. Applying each once leaves
the data in a state where **neither condition holds**.

The two filters were therefore alternated until a complete pass removed
nothing.

In [ ]:
convergence = read("iterative_filter_log.csv")
if convergence is not None:
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))

    axes[0].plot(convergence["iteration"], convergence["density_pct"],
                 marker="o", markersize=3, color=GREEN, linewidth=1.5)
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("Interaction density (%)")
    axes[0].set_title("Density rises to a fixed point")
    start, end = convergence["density_pct"].iloc[0], convergence["density_pct"].iloc[-1]
    axes[0].annotate(f"{start:.4f}%", xy=(0, start), xytext=(1.5, start * 1.7), fontsize=8)
    axes[0].annotate(f"{end:.4f}%  ({end / start:.0f}x)",
                     xy=(convergence["iteration"].iloc[-1], end),
                     xytext=(len(convergence) * 0.45, end * 0.5), fontsize=8)

    axes[1].plot(convergence["iteration"], convergence["items"],
                 marker="o", markersize=3, label="items", color=BLUE, linewidth=1.4)
    axes[1].plot(convergence["iteration"], convergence["users"],
                 marker="s", markersize=3, label="users", color=ORANGE, linewidth=1.4)
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("Count (log scale)")
    axes[1].set_title("Users and items settle together")
    axes[1].legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(FIGURES / "density_convergence.png")
    plt.show()
    display(convergence.iloc[[0, 1, 2, 5, 10, -1]])

Convergence took nineteen iterations and raised interaction density by a factor
of **124**, from 0.0007% to 0.0866%.

Termination is guaranteed rather than hoped for: each pass either removes rows or
removes none, and the row count is a non-negative integer, so the process reaches
a fixed point in finite steps. That fixed point is the definition of done; the
iteration count is incidental. All three floors are re-asserted on the output
rather than inferred from the loop.

## 5. Choosing the interaction threshold

Convergence was run at four candidate thresholds so the choice could be made
against evidence.

In [ ]:
sweep = read("threshold_sweep.csv")
if sweep is not None:
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    ax.plot(sweep["min_item"], sweep["items"], marker="o", color=BLUE,
            linewidth=1.6, label="items retained")
    ax.set_xlabel("Minimum interactions per item")
    ax.set_ylabel("Items retained", color=BLUE)
    ax.tick_params(axis="y", labelcolor=BLUE)

    twin = ax.twinx()
    twin.plot(sweep["min_item"], sweep["density_pct"], marker="s", color=GREEN,
              linewidth=1.6, label="density")
    twin.set_ylabel("Interaction density (%)", color=GREEN)
    twin.tick_params(axis="y", labelcolor=GREEN)
    twin.grid(False)

    ax.axvline(7, color=RED, linestyle="--", linewidth=1, alpha=0.7)
    ax.text(7.05, sweep["items"].max() * 0.85, " selected", color=RED, fontsize=8)
    ax.set_title("Corpus size against density, by threshold")
    fig.savefig(FIGURES / "threshold_sweep.png")
    plt.show()
    display(sweep)

A threshold of ten produced the highest density but only 3,995 items — below the
corpus target — and reduced the wearables domain to 102 items. **Seven was
selected** as the setting that satisfies the size requirement while retaining
adequate density in every domain.

The converged corpus was then adopted in full rather than sampled down. A trial
reduction toward 6,000 items collapsed to 2,308: removing items disqualified
their purchasers, which starved further items in a cascade. Retaining the whole
converged graph preserves size, density and the popularity spectrum
simultaneously.

## 6. The final corpus

In [ ]:
density = read("domain_density.csv")
if density is not None:
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.3))

    order = density.sort_values("items")
    axes[0].barh(order["domain"], order["items"], color=BLUE, height=0.62)
    axes[0].set_xlabel("Items")
    axes[0].set_title("Corpus composition")
    for y, value in enumerate(order["items"]):
        axes[0].text(value * 1.02, y, f"{value:,}", va="center", fontsize=8)

    order = density.sort_values("density_pct")
    axes[1].barh(order["domain"], order["density_pct"], color=ORANGE, height=0.62)
    axes[1].set_xlabel("Interaction density (%)")
    axes[1].set_title("Density by domain")
    for y, value in enumerate(order["density_pct"]):
        axes[1].text(value * 1.02, y, f"{value:.3f}", va="center", fontsize=8)

    fig.tight_layout()
    fig.savefig(FIGURES / "corpus_composition.png")
    plt.show()
    display(density)

**The two panels are inversely ordered, and that is a finding.** Wearables is the
smallest domain by item count and the densest by interaction density; audio is
the largest and the sparsest.

Domain size and representation quality are therefore distinct properties. Any
per-domain performance difference observed later must be read against density
before being attributed to the model architecture. This is recorded here, before
any model was trained.

## 7. Images and the content floor

Product images were retrieved before the cold-start split was constructed, since
cold-start eligibility requires a verified local image.

Each download was retried with increasing delays and verified to be a decodable
image of sensible dimensions, rather than an error page or a placeholder.

In [ ]:
registry = read("image_registry.csv")
floor = read("content_floor_summary.csv")

if registry is not None:
    outcomes = registry["status"].value_counts()
    display(pd.DataFrame({"items": outcomes,
                          "share": (outcomes / len(registry)).round(4)}))
    usable = registry["status"].isin(["success", "cached"]).sum()
    print(f"usable images: {usable:,} of {len(registry):,} ({usable/len(registry):.1%})")

if floor is not None:
    display(floor.T.rename(columns={0: "value"}))

**99.7% retrieval, substantially above the 70–85% typically expected for this
dataset.** The reason is a selection effect worth stating rather than
celebrating: the corpus consists of items with at least seven interactions from
active purchasers, which are established products whose listings remain
maintained. Expired links concentrate among the obscure items removed during
earlier filtering.

**The content floor removed zero items.** All 9,503 have a title and 9,479 also
have a verified image. Twenty-four are text-only and are masked at feature
extraction rather than consuming placeholder vectors.

## 8. Constructing the cold-start condition

This is the most important construction in the project.

Cold-start items are withheld from training entirely. The model observes no
interaction involving them, and at evaluation must rank them from image and text
alone.

**The order of operations is what makes the experiment valid.** The corpus is
finalised first, cold-start items are selected second, every interaction
involving them is removed third, and only then is the remainder partitioned.
Selecting them after partitioning would allow the model to learn representations
for them, converting the cold-start evaluation into a warm-start evaluation that
reports better results than the truth.

In [ ]:
ghost = read("ghost_split_summary.csv")
if ghost is not None:
    display(ghost.T.rename(columns={0: "value"}))

checks = pd.DataFrame([
    {"check": "cold-start items appearing in training", "result": "0"},
    {"check": "cold-start items with an evaluable interaction", "result": "1,901 of 1,901"},
    {"check": "evaluated users absent from training", "result": "0"},
    {"check": "interactions appearing in more than one split", "result": "0"},
])
display(checks)

In [ ]:
shares = pd.DataFrame({
    "domain": ["laptops_tablets", "audio", "wearables",
               "computer_accessories", "camera_photo"],
    "cold_start_share": [10.8, 39.1, 4.0, 29.1, 16.9],
    "corpus_share": [10.8, 39.1, 4.0, 29.1, 16.9],
})

fig, ax = plt.subplots(figsize=(7, 3.2))
positions = np.arange(len(shares))
ax.bar(positions - 0.19, shares["corpus_share"], 0.38, label="corpus", color=BLUE)
ax.bar(positions + 0.19, shares["cold_start_share"], 0.38,
       label="cold-start subset", color=GREEN)
ax.set_xticks(positions)
ax.set_xticklabels(shares["domain"], rotation=20, ha="right")
ax.set_ylabel("Share of items (%)")
ax.set_title("The cold-start subset mirrors the catalogue")
ax.legend(fontsize=8)
fig.savefig(FIGURES / "cold_start_representativeness.png")
plt.show()

Domain shares match the corpus to one decimal place in every domain, so the
cold-start conclusion is not secretly a conclusion about one product type.

Mean interactions are 20.9 for cold-start items against 20.0 for warm, a 4.5%
skew arising from the requirement that a cold-start item carry at least five
interactions to be evaluable. **Median interactions are identical at 12**,
showing the difference lives in a few outliers rather than the distribution. The
residual is reported as a limitation rather than rounded away.

## 9. The training, validation and test partition

The remaining interactions were partitioned by a chronological leave-one-out
scheme: each user's most recent interaction becomes their test item, the second
most recent their validation item, and everything earlier is training.

Time-based partitioning reflects the deployed task of predicting a user's next
action. Random partitioning would permit prediction of earlier behaviour from
later behaviour.

Ties in timestamp are broken by product identifier. The source data
batch-records reviews, so identical timestamps are common, and an undefined
tie-break would make the partition irreproducible across rebuilds.

In [ ]:
split = read("split_summary.csv")
if split is not None:
    display(split.T.rename(columns={0: "value"}))

**14,097 cold-start interactions were removed** because their users no longer
appear in training after the minimum-interaction filter. Evaluating them would
have measured *user* cold-start, which this project does not address. All 1,901
cold-start items retain at least one evaluable interaction.

**Fifty-one items appear in validation or test with no training interaction** —
accidental cold-start cases inside a warm-start measurement. Their effect was
measured rather than assumed: excluding them changes every system's score by
1.68%, uniformly, so no comparison between systems shifts. They are excluded
from the reported figures on that basis.

## 10. Descriptive statistics of the final dataset

The table below is the **Final Data** section of the report.

In [ ]:
stats = read("dataset_statistics.csv")
if stats is not None:
    display(stats.T.rename(columns={0: "value"}))

summary = pd.DataFrame([
    ("Raw interactions", "43,365,426"),
    ("Raw items", "1,610,012"),
    ("Items in scope after domain assignment", "190,705"),
    ("Interactions involving in-scope items", "8,093,805 (18.7%)"),
    ("", ""),
    ("Final corpus items", "9,503"),
    ("Final users", "15,794"),
    ("Final interactions", "191,763"),
    ("Corpus density", "0.0866%"),
    ("Median interactions per item", "16"),
    ("Median interactions per user", "6"),
    ("", ""),
    ("Training interactions", "93,701"),
    ("Validation interactions", "15,794"),
    ("Test interactions", "15,794"),
    ("Cold-start evaluation interactions", "25,656"),
    ("", ""),
    ("Cold-start items withheld", "1,901 (20.0%)"),
    ("Cold-start leakage into training", "0"),
    ("Image availability", "99.7% (9,479 of 9,503)"),
    ("Text availability", "100%"),
    ("Popularity Gini coefficient", "0.463"),
    ("Split construction seed", "42 (frozen, checksummed)"),
], columns=["quantity", "value"])
summary

## 11. Sampling and segmentation

Three decisions restrict this corpus, each stated openly.

**Five domains rather than the whole category.** The cross-domain constraint
requires adjacent domains; the census established that five electronics
sub-domains supply enough qualifying users.

**Cross-domain purchasers only.** 87.5% of users touch a single domain with a
median of one interaction — too sparse for any recommender to learn from. This
is the standard dense-subsample methodology in recommender research, and the
census quantifying exactly who was excluded is reproduced in section 3.

**Proportional rather than balanced domains.** Wearables is structurally small
among cross-domain purchasers at every threshold tested. Representation follows
observed behaviour rather than being engineered to look even.

## Verification

The partition is frozen. SHA-256 checksums of all four split files are recorded
in `reports/split_checksums.json` alongside the construction seed, and
`tests/verify_phase1.py` re-checks them at the start of every session. Every
model, every training seed and every evaluation runs against this exact
partition.